我已经只读检查了当前 Godot 项目、剧情 JSON 和开发日志，没有修改任何内容。

结论是：项目目前不需要“大型框架式重构”，而需要先建立一个稳定的剧情运行核心，把目前集中在 `main_scene.gd` 中的“读取、解释、推进、跳转、演出、UI 控制”拆开。否则一旦加入调查、证言、举证和存档，主脚本会迅速失控。

## 一、当前项目的主要问题

当前核心调用链是：

```text
main_scene.gd
├── 读取 JSON
├── 维护剧情索引
├── 解释 JSON 命令
├── 查找 anchor
├── 处理选择
├── 切换剧本
├── 控制背景
├── 控制角色
├── 控制转场
├── 控制输入
└── 控制播放节奏
```

问题并非单纯“文件放得乱”，而是职责边界还没有形成。

### 1. `main_scene.gd` 承担了过多职责

[main_scene.gd](</D:/projects/godot-projects/a-逆转裁判/scripts/main_scene.gd>) 同时充当了：

- 剧情文件加载器
- 剧情解释器
- 剧情播放器
- 输入控制器
- 场景导演
- 分支跳转器
- 背景控制器
- 转场协调器

目前只有少数命令时还能维护；加入 `if`、变量、证物、调查点、证言、追问、举证以后，这里会出现非常大的 `if line.has(...)` 链。

### 2. 剧情逻辑与表现层直接耦合

例如剧情处理器直接调用：

```gdscript
background.texture = load(...)
character_sprite.change_character(...)
dialog_ui.change_line(...)
SceneManager.transition_out(...)
```

这意味着“剧情运行”无法脱离当前场景树进行测试，也无法在调查模式和法庭模式中复用。

更理想的结构是：

```text
剧情播放器
    ↓ 发出语义事件
场景导演
    ↓ 调用
背景、角色、对话框、声音、转场
```

剧情播放器只应该知道“现在要显示一句对白”或“现在要执行 background 命令”，不应该知道具体 `TextureRect` 在哪里。

### 3. `dialog_ui.gd` 同时处理 UI、打字机和选择逻辑

[dialog_ui.gd](</D:/projects/godot-projects/a-逆转裁判/scripts/dialog_ui.gd>) 现在负责：

- 对话文字显示
- 说话人显示
- 打字机计时
- 标点暂停
- 打字音效
- 动态创建选项
- 输入焦点

短期没有问题，但后续要加入文本标签、回看、自动播放、已读快进和句内命令时会变得很复杂。

至少应该逐步拆成：

```text
DialogueView       对话框整体接口
├── Typewriter     逐字显示和句内事件
├── ChoiceView     选项生成与选择
└── BlipPlayer     打字音效
```

### 4. 角色数据被硬编码在脚本中

[Character.gd](</D:/projects/godot-projects/a-逆转裁判/scripts/Character.gd>) 中用枚举和字典保存角色信息：

```gdscript
enum Name { APOLLO, PHOENIX, TRUCY }
const CHARACTER_DETAILS = {...}
```

这会导致每添加角色都必须修改代码。与此同时，剧情 JSON 使用字符串角色名，两端还需要手动转换。

更适合使用：

```text
CharacterDefinition.tres
├── id
├── display_name
├── blip_sound
├── name_color
└── SpriteFrames
```

角色 ID 使用稳定的 `StringName`，例如：

```text
phoenix
apollo
trucy
```

显示名可以本地化，不能拿显示名充当程序 ID。

### 5. JSON 当前只是“未验证的 Dictionary 数组”

例如同一个 `expression` 在两个剧本中的语义并不一致：

```json
"expression": "confident-talking"
```

另一个文件则是：

```json
"expression": "confident"
```

而 `character_sprite.gd` 又会自动拼接 `-talking` 或 `-idle`。这说明数据格式与显示逻辑之间还没有正式契约。

目前还存在以下风险：

- `speaker`、`text` 等字段运行时才发现缺失。
- `goto` 指向不存在的 anchor 时没有可靠返回值。
- anchor 每次跳转都线性遍历。
- 一个 Dictionary 可以意外同时包含多个相互冲突的命令。
- `next_scene` 实际切换的是剧情文件，不是 Godot 场景。
- 资源路径通过字符串拼接，重命名后容易失效。
- JSON 无法表达明确的命令类型和参数类型。

### 6. `SceneManager` 名称与职责不一致

[SceneManager.gd](</D:/projects/godot-projects/a-逆转裁判/scripts/SceneManager.gd>) 当前并不管理 Godot 场景，主要负责全屏转场。因此更准确的名称应该是：

```text
TransitionLayer
或
TransitionService
```

它作为 Autoload 并非完全错误，但目前通过代码创建全部 UI 节点，后续增加闪白、异议切入、法庭滑屏等演出后不易在编辑器里调整。更合适的方式是让它成为带 `CanvasLayer`、`ColorRect` 和 `AnimationPlayer` 的场景。

### 7. 目录中已经出现命名和临时文件问题

例如：

- `spirites` 应为 `sprites`
- `phonix__sprite_frames.tres` 应统一为 `phoenix_sprite_frames.tres`
- `Character.gd` 与其他脚本的命名规则不一致
- `character.tscn5906990066.tmp` 是遗留临时文件
- 中英文路径、空格、大小写和双下划线混用

这些现在影响不大，但制作自定义编译器、自动资源检索和导出时会持续制造边缘问题。

---

# 二、建议的目标架构

建议采用四层结构：

```text
内容数据
    ↓
剧情编译/加载
    ↓
运行时与玩法规则
    ↓
Godot 表现层
```

具体关系：

```text
DrameX 源文件
      │
      ▼
DrameXCompiler / JSON Loader
      │
      ▼
StoryProgram（规范化中间数据）
      │
      ▼
StoryRunner ─────────────── GameState
      │                         │
      │ 产生 Action             │ 保存变量、证物、进度
      ▼                         │
PresentationDirector ◄──────────┘
      │
      ├── DialogueView
      ├── CharacterStage
      ├── BackgroundView
      ├── AudioDirector
      ├── TransitionLayer
      └── Court/Investigation UI
```

最重要的边界是：

- `StoryRunner` 不引用任何 Label、Sprite、TextureRect。
- UI 不读取和修改剧情索引。
- `GameState` 不控制动画。
- DrameX 编译器不加载 Godot 场景。
- 玩法系统通过正式命令进入剧情运行器，不把特殊逻辑塞回主循环。

## 三、建议的项目目录

```text
res://
├── app/
│   ├── game.tscn
│   ├── game.gd
│   ├── game_state.gd
│   └── game_mode.gd
│
├── story/
│   ├── runtime/
│   │   ├── story_program.gd
│   │   ├── story_runner.gd
│   │   ├── story_frame.gd
│   │   ├── story_action.gd
│   │   └── expression_evaluator.gd
│   ├── loading/
│   │   ├── story_loader.gd
│   │   ├── story_validator.gd
│   │   └── json_story_loader.gd
│   ├── dramex/
│   │   ├── tokenizer.gd
│   │   ├── parser.gd
│   │   ├── compiler.gd
│   │   └── diagnostics.gd
│   └── content/
│       └── cases/
│           └── case_01/
│               ├── case_manifest.tres
│               ├── investigation_01.dramex
│               └── trial_01.dramex
│
├── presentation/
│   ├── presentation_director.gd
│   ├── dialogue/
│   │   ├── dialogue_view.tscn
│   │   ├── dialogue_view.gd
│   │   ├── typewriter.gd
│   │   └── choice_view.tscn
│   ├── characters/
│   │   ├── character_stage.tscn
│   │   └── character_stage.gd
│   ├── backgrounds/
│   ├── audio/
│   └── transitions/
│       ├── transition_layer.tscn
│       └── transition_layer.gd
│
├── gameplay/
│   ├── common/
│   │   ├── evidence.gd
│   │   ├── profile.gd
│   │   ├── court_record.gd
│   │   └── penalty_state.gd
│   ├── investigation/
│   │   ├── investigation_controller.gd
│   │   ├── location.gd
│   │   ├── examine_point.gd
│   │   └── investigation_view.tscn
│   ├── trial/
│   │   ├── trial_controller.gd
│   │   ├── testimony.gd
│   │   ├── testimony_statement.gd
│   │   ├── cross_examination.gd
│   │   └── trial_view.tscn
│   └── mechanics/
│       ├── psyche_lock/
│       ├── perceive/
│       ├── mood_matrix/
│       ├── logic_chess/
│       └── forensic/
│
├── data/
│   ├── characters/
│   │   ├── phoenix/
│   │   │   ├── phoenix.tres
│   │   │   ├── phoenix_sprite_frames.tres
│   │   │   └── sprites/
│   │   └── trucy/
│   ├── evidence/
│   ├── locations/
│   ├── audio/
│   └── themes/
│
├── ui/
│   ├── menus/
│   ├── court_record/
│   ├── backlog/
│   └── settings/
│
├── persistence/
│   ├── save_service.gd
│   ├── save_data.gd
│   └── settings_service.gd
│
├── tests/
├── addons/
└── project.godot
```

不需要现在一次性创建全部目录。它代表长期归属规则。

---

# 四、当前文件应如何迁移职责

这是概念映射，不是要求立即搬文件。

| 当前内容 | 目标职责 |
|---|---|
| `main_scene.gd` 的 JSON 读取 | `JsonStoryLoader` |
| `main_scene.gd` 的索引、goto、anchor | `StoryRunner` |
| `main_scene.gd` 的命令判断 | `StoryAction` / action handlers |
| `main_scene.gd` 的节点协调 | `PresentationDirector` |
| `main_scene.gd` 的玩家输入 | `Game` 或当前模式 Controller |
| `dialog_ui.gd` 的文字动画 | `Typewriter` |
| `dialog_ui.gd` 的选项 | `ChoiceView` |
| `dialog_ui.gd` 的外部 API | `DialogueView` |
| `Character.gd` 的字典 | `CharacterDefinition` Resources |
| `character_sprite.gd` | `CharacterStage` 或 `CharacterActor` |
| `text_blip_sound.gd` | 保留为 `BlipPlayer` 子组件 |
| `SceneManager.gd` | `TransitionLayer` |
| `first_scene.json` 等 | 经验证的 `StoryProgram` 数据 |

## 五、剧情运行核心应该怎样设计

### 1. 使用明确的运行时 Action

不要让整个系统长期依赖原始 Dictionary。JSON 或 DrameX 读入后，先变成规范化动作：

```text
DialogueAction
ChoiceAction
JumpAction
CallAction
ReturnAction
BackgroundAction
ShowCharacterAction
HideCharacterAction
MusicAction
TransitionAction
SetVariableAction
ConditionalAction
WaitAction
GameplayAction
```

第一版不一定需要每个动作一个文件；可以先用一个带 `type` 和正式字段的 `StoryAction`。关键是加载后完成验证，播放器不再反复询问：

```gdscript
if line.has("speaker")
if line.has("goto")
if line.has("location")
```

### 2. `StoryRunner` 只维护剧情执行状态

建议状态包括：

```text
current_program_id
instruction_pointer
call_stack
variables
waiting_state
current_mode
```

公开接口保持很小：

```gdscript
load_program(program)
start(entry_label)
advance()
choose(choice_id)
resume()
create_snapshot()
restore_snapshot(snapshot)
```

它向外产生：

```text
action_requested(action)
choices_requested(choices)
story_paused
story_finished
runtime_error(diagnostic)
```

表现层完成异步动画后调用 `resume()`。

### 3. 预先构建标签表

加载剧情时生成：

```gdscript
labels = {
    &"start": 0,
    &"court_cases": 17,
    &"ending": 29
}
```

同时验证：

- 标签是否重复
- `goto` 目标是否存在
- `call` 文件是否存在
- 角色是否注册
- 表情是否存在
- 参数类型是否正确

这样跳转为 O(1)，错误也会在启动或构建时出现，而不是玩家玩到一半才出现。

### 4. 区分三个容易混淆的概念

建议从现在起固定语义：

- `background`：更换背景图片
- `jump/call`：切换剧情程序或剧情位置
- `change_scene`：切换真正的 Godot 场景

你的 `next_scene` 当前实际代表“切换剧情文件”，不应继续称为 scene。DrameX 手册中的 `call`、`jump`、`return` 已经比现有 JSON 的 `next_scene` 更合理。

### 5. DrameX 先定义中间表示，再写语法解析器

你的 [DrameX 手册](</D:/OneDrive/log/Godot开发日志/DrameX手册v0.1.0.ipynb>) 已经覆盖：

- 对话与旁白
- 标签
- `goto/call/jump/return`
- 条件和变量
- 选择
- 场景演出命令
- 句内效果
- 自定义组合命令

但不建议现在立刻实现完整语言。

正确顺序是：

```text
手写规范化 JSON
→ StoryProgram/StoryAction 跑通
→ Validator
→ DrameX 最小语法
→ 编译成相同的 StoryProgram
→ 编辑器工具
```

否则你会同时调试语言语法、编译器、运行器和玩法系统，定位问题会非常困难。

此外，“编译器自行检索资源”不应只按文件名模糊搜索。建议由构建步骤生成资源注册表：

```text
phoenix → res://data/characters/phoenix/phoenix.tres
courtroom → res://data/locations/courtroom.tres
objection → res://data/audio/sfx/objection.ogg
```

发现重复 ID 时直接报告构建错误。

---

# 六、逆转裁判玩法的系统规划

不建议把“调查”和“法庭”做成两个庞大的 State 脚本。更合适的是：

```text
Game
└── 当前 GameMode
    ├── StoryMode
    ├── InvestigationMode
    ├── CrossExaminationMode
    └── SpecialMechanicMode
```

每个模式负责：

- 接受哪些输入
- 显示哪些 UI
- 怎样结束并返回剧情
- 怎样写入 `GameState`

剧情系统通过类似命令进入玩法：

```text
@investigation investigation_day_1
@cross_examination witness_testimony_1
@psyche_lock maya_secret
```

玩法结束后返回结果：

```text
success
failure
cancelled
selected_evidence
selected_statement
```

## 七、核心玩法模块

### 1. 法庭记录 Court Record

这是调查和庭审共同依赖的基础，应优先实现。

数据模型：

```text
CourtRecord
├── evidence[]
├── profiles[]
└── revision

Evidence
├── id
├── display_name
├── description
├── icon
├── detail_image
├── acquired
└── metadata
```

必须支持：

- 获得证物
- 更新证物说明
- 删除/替换证物
- 人物档案
- 从对话、调查和证言界面打开
- 向当前玩法返回玩家选择的条目

不要让 Court Record 自己判断证物是否正确；正确性由调用它的玩法系统判断。

### 2. 调查模式

一处调查场景可以建模为：

```text
InvestigationLocation
├── background
├── available_destinations
├── examine_points
├── people
├── talk_topics
├── present_reactions
└── unlock_conditions
```

四个基础操作：

- 调查 `examine`
- 交谈 `talk`
- 指证/出示 `present`
- 移动 `move`

需要支持：

- 可点击调查热点
- 调查热点状态和一次性事件
- 新话题解锁
- 新地点解锁
- 向人物出示证物
- 默认出示反应
- 完成条件
- 调查阶段结束判定

关键设计：这些操作的结果尽量是“调用一段剧情”，不要把大量对白写在调查控制器里。

### 3. 证言与询问

核心数据结构：

```text
Testimony
├── statements[]
├── penalty
└── completion_rule

TestimonyStatement
├── id
├── text
├── speaker
├── press_script
├── accepted_evidence[]
├── wrong_present_script
└── replacement/insertion rules
```

玩法循环：

```text
展示证言
→ 左右浏览陈述
→ 追问或举证
→ 追问可能修改证言
→ 举证验证
→ 正确则进入剧情
→ 错误则扣除惩罚值并返回询问
```

应支持：

- 前后浏览证言
- 当前陈述索引
- 追问
- 举证
- 举证人物档案
- 错误举证
- 生命/惩罚槽
- 追加、替换或删除证言
- 证言循环
- 中途保存与恢复

不要把“正确证物 ID”写在按钮代码中，应放在 `TestimonyStatement` 数据里。

### 4. 惩罚系统

建议做成共享运行状态，而不是法庭 UI 的字段：

```text
PenaltyState
├── current
├── maximum
├── damage(amount)
├── restore(amount)
└── depleted
```

这样它也可以服务于：

- 普通错误举证
- 心灵枷锁失败
- 特殊推理失败
- 案件难度配置

### 5. 存档、读档与剧情回看

存档不能最后才设计，因为它会反向约束所有运行状态。

至少需要保存：

```text
版本号
案件 ID
当前模式
剧情程序 ID
指令位置或稳定 checkpoint
call stack
剧情变量
已读文本集合
证物和人物档案
调查地点状态
已解锁话题
当前证言及陈述位置
惩罚值
特殊机制状态
当前背景、角色、音乐
```

不要直接序列化整个节点树。存档只保存纯数据，读档后由各表现系统恢复画面。

同时实现：

- 自动存档
- 手动存档
- 存档版本迁移
- 对话历史
- 已读文本标记
- 仅已读快进
- 自动播放

## 八、系列特殊玩法如何处理

“逆转裁判所有玩法”跨度很大。建议将系列通用玩法作为核心，把每代的特殊系统作为可插拔玩法模块。

优先级可分为：

### 核心必做

- 普通剧情
- 选择
- 调查、交谈、移动、出示
- 法庭记录
- 证言
- 追问
- 举证
- 惩罚槽
- 存读档
- 对话回看与快进

### 第二阶段机制

- 心灵枷锁
- 看破
- 逻辑推理/逻辑组合
- 现场重现或 3D/2D 证物检查
- 科学调查小游戏

### 后期独立机制

- 情绪矩阵
- 逻辑象棋
- 水镜/御魂托宣
- 多角度现场调查
- 指纹、鲁米诺、足迹等法证小游戏
- 最终推理图解
- 特定案件的一次性小游戏

统一接口可以很小：

```text
SpecialMechanic
├── start(context)
├── handle_input(action)
├── create_snapshot()
├── restore_snapshot(data)
└── finished(result)
```

这样特殊玩法不需要进入 `StoryRunner` 的主循环，只需要由故事命令启动并等待结果。

---

# 九、推荐开发阶段

## 阶段 0：固定规则与命名

目标：在重构前消除概念歧义。

完成标准：

- 确定 `scene`、`story program`、`chapter`、`location` 的含义。
- 固定角色、证物、地点、表情的 ID 规则。
- 定义规范化剧情 Action 格式。
- 决定哪些对象可以成为 Autoload。
- 为 DrameX 写一个“第一版不支持什么”的列表。

建议只保留这些全局服务：

```text
GameState
SaveService
SettingsService
```

转场、对话、角色显示不必成为全局管理器。

## 阶段 1：在功能不变的前提下拆分当前系统

目标：让现有两个 JSON 剧情完整运行，但主场景不再解释所有命令。

实施顺序：

1. 抽出 `JsonStoryLoader`。
2. 加入剧情验证和标签索引。
3. 抽出 `StoryRunner`。
4. 建立 `PresentationDirector`。
5. 把选择界面从打字机职责中分离。
6. 把角色字典转换为资源数据。
7. 把 `SceneManager` 收窄为转场组件。
8. 最后让 `main_scene.gd` 只负责装配和启动。

完成后的 `main_scene.gd` 理想上只做：

```text
获取节点
连接信号
加载初始剧情
启动播放器
处理退出/暂停
```

## 阶段 2：建立可扩展的剧情中间表示

目标：运行器不再直接消费随意结构的 Dictionary。

完成标准：

- 每条 Action 有明确类型。
- 所有标签在加载阶段解析。
- 所有跳转提前验证。
- 缺失角色、背景或表情能给出文件与行号。
- `call/jump/return` 具备稳定行为。
- 条件、变量和选择不依赖 UI 实现。

## 阶段 3：完成一个“调查日”垂直切片

只制作一个很短但完整的流程：

```text
进入事务所
→ 与人物交谈
→ 解锁话题
→ 移动到现场
→ 调查热点
→ 获得证物
→ 向人物出示证物
→ 完成调查
```

完成标准：

- 数据驱动，不在 Controller 中硬编码具体案件。
- 中途保存退出后可以恢复。
- 调查玩法结束后能继续同一个故事运行栈。

## 阶段 4：完成一个“庭审日”垂直切片

流程：

```text
开庭剧情
→ 证言
→ 追问
→ 追加一条陈述
→ 在新陈述上举证
→ 错误举证扣血
→ 正确后推进剧情
→ 判决
```

这一阶段完成后，游戏已经具备逆转裁判的核心闭环。

## 阶段 5：存档、回看和玩家体验

加入：

- 存档/读档
- 自动存档
- 对话历史
- 自动播放
- 跳过已读
- 文本速度
- 音量与按键设置
- 窗口适配
- 无鼠标操作和焦点管理

这一阶段要早于大规模写案件内容。

## 阶段 6：实现 DrameX MVP

第一版只支持：

```text
对话、旁白
标签
goto/call/jump/return
choice
if/else
变量
background/music/show/hide
wait/transition/sound
启动调查
启动证言
```

暂缓：

- 任意自定义函数
- 递归宏
- 复杂类型系统
- 插件 API
- 完整 IDE 实时预览
- 模糊资源自动检索

编译输出应与阶段 2 的 `StoryProgram` 完全一致。这样 JSON 和 DrameX 可以共用同一运行器。

## 阶段 7：特殊玩法插件

建议顺序：

1. 心灵枷锁
2. 证物细节检查和科学调查
3. 看破
4. 逻辑组合/最终推理
5. 情绪矩阵
6. 逻辑象棋
7. 水镜等作品专属机制

每完成一个机制，都制作一个独立演示场景和保存恢复测试。

## 阶段 8：内容生产工具

最后再做：

- DrameX VS Code 语法高亮
- 标签跳转
- 自动补全
- 角色、表情和资源 ID 检查
- 编译诊断
- 剧情图预览
- Godot 导入插件
- 一键运行当前标签

---

# 十、最重要的三项近期任务

如果从当前进度继续，我建议严格按以下顺序：

1. **先定义 StoryAction 和 StoryProgram。**  
   不先写 DrameX Parser，也不先加入新玩法。

2. **把 `main_scene.gd` 拆成 Loader、Runner、Director。**  
   保证现有剧情表现完全不变。

3. **用“一个调查日 + 一个证言”验证架构。**  
   只有这两个核心玩法顺利跑通，才开始设计所有特殊机制。

开发日志中提出的“Parser → Player → Command”方向是正确的；但“为每个系统都建立全局 Manager”需要收缩。这个项目真正需要的是一个稳定的运行时、少量全局状态，以及按玩法生命周期实例化的 Controller。这样后续新增机制是增加模块，而不是不断扩大 `main_scene.gd`。

<hr style="border: none; height: 2px; background-color: red
    ;">

可以把 Godot 的代码组织理解为一句话：

> 场景负责组合对象，脚本负责对象行为，Resource 负责数据，信号负责向外通知，上层节点负责协调。

这比机械套用 MVC、ECS 或“Manager 万能单例”更符合 Godot 的工作方式。

## 一、Godot 项目组织的最佳实践

### 1. 按“功能/领域”组织目录

不建议把整个项目简单拆成：

```text
scripts/
scenes/
textures/
audio/
```

项目变大后，一个角色的场景、脚本、动画和贴图会散落在四处。Godot 官方更推荐把资源放在使用它们的场景附近。[项目组织文档](https://docs.godotengine.org/en/stable/tutorials/best_practices/project_organization.html)

视觉小说可以这样组织：

```text
res://
├── core/
│   ├── game.gd
│   ├── scene_router.gd
│   └── save_service.gd
├── dialogue/
│   ├── dialogue_runner.gd
│   ├── dialogue_line.gd
│   ├── dialogue_choice.gd
│   └── dialogue_box/
│       ├── dialogue_box.tscn
│       ├── dialogue_box.gd
│       └── dialogue_box_theme.tres
├── characters/
│   ├── character_data.gd
│   └── alice/
│       ├── alice.tres
│       └── portraits/
├── chapters/
│   └── chapter_01/
│       ├── chapter_01.tscn
│       ├── chapter_01.gd
│       ├── dialogue.json
│       └── backgrounds/
├── menus/
│   ├── main_menu/
│   ├── settings_menu/
│   └── save_menu/
├── shared/
│   ├── audio/
│   ├── themes/
│   └── transitions/
├── addons/
└── project.godot
```

`core` 只放真正跨越整个游戏生命周期的东西；具体角色、章节和 UI 应各自形成独立模块。

命名方面，Godot 官方建议：

- 文件和目录使用 `snake_case`
- 节点名使用 `PascalCase`
- 第三方插件放入顶层 `addons/`
- 路径大小写保持一致，避免 Windows 正常、导出后失败

### 2. 一个场景代表一个能够独立理解的游戏概念

适合作为独立场景的对象包括：

- 对话框
- 角色立绘显示器
- 选项列表
- 主菜单
- 存档面板
- 单独章节
- 转场效果

理想状态是：把 `dialogue_box.tscn` 单独运行或实例化到另一个项目中，它依然能够工作，而不是偷偷依赖某个固定父节点。

Godot 官方特别强调场景应尽量自包含；如果确实需要外部对象，应由父级注入依赖或连接信号。[场景组织文档](https://docs.godotengine.org/en/stable/tutorials/best_practices/scene_organization.html)

判断是否应该拆场景，可以问：

1. 这个对象能否独立命名？
2. 是否会被重复实例化？
3. 是否包含自己的节点、资源和行为？
4. 修改它时，是否不应该影响其他系统？

多数回答为“是”，就适合成为独立场景。

### 3. 子节点只描述“发生了什么”，不要决定整个游戏接下来做什么

例如，对话框不应该直接切换章节：

```gdscript
# 不推荐
func _on_next_button_pressed() -> void:
    get_tree().change_scene_to_file("res://chapters/chapter_02.tscn")
```

更好的方式是发出信号：

```gdscript
class_name DialogueBox
extends Control

signal advance_requested
signal choice_selected(choice_id: StringName)

func _on_next_button_pressed() -> void:
    advance_requested.emit()
```

然后由拥有它的上层对象决定行为：

```gdscript
func _ready() -> void:
    dialogue_box.advance_requested.connect(dialogue_runner.advance)
    dialogue_box.choice_selected.connect(dialogue_runner.select_choice)
```

这样对话框不知道：

- 谁在管理故事
- 下一章是什么
- 是否需要存档
- 按钮点击后是否播放转场

它只负责 UI。信号正是 Godot 用来降低对象间耦合的核心机制。[信号文档](https://docs.godotengine.org/en/stable/getting_started/step_by_step/signals.html)

不过也不要把所有方法调用都改成信号：

- **命令某个对象做事**：直接调用方法
- **通知外界某件事已经发生**：发出信号

例如：

```gdscript
dialogue_box.show_line(line) # 上层发出命令
line_finished.emit()         # 对话框报告事件
```

### 4. 由共同的父节点协调兄弟节点

假设场景结构如下：

```text
Chapter
├── DialogueRunner
├── DialogueBox
├── CharacterDisplay
├── BackgroundDisplay
└── MusicPlayer
```

不要让 `DialogueBox` 自己寻找 `CharacterDisplay`，也不要大量使用：

```gdscript
get_parent().get_node("../CharacterDisplay")
```

应该由 `Chapter` 或专门的 `SceneDirector` 保存引用、建立连接并协调它们：

```gdscript
class_name SceneDirector
extends Node

@onready var dialogue_runner: DialogueRunner = %DialogueRunner
@onready var dialogue_box: DialogueBox = %DialogueBox
@onready var character_display: CharacterDisplay = %CharacterDisplay

func _ready() -> void:
    dialogue_runner.line_started.connect(_on_line_started)
    dialogue_runner.character_changed.connect(character_display.show_character)

func _on_line_started(line: DialogueLine) -> void:
    dialogue_box.show_line(line)
```

原则是：

> 子节点了解自己；父节点了解并协调直接子节点；兄弟节点尽量不要互相了解。

### 5. 把故事数据和表现逻辑分离

不要在 UI 脚本中塞满对白：

```gdscript
if current_line == 4:
    label.text = "你好"
    portrait.texture = alice_happy
```

对白、人物和分支应该是数据。Godot 中可以使用自定义 `Resource`：

```gdscript
class_name DialogueLine
extends Resource

@export var speaker: CharacterData
@export_multiline var text: String
@export var expression: StringName
@export var next_line_id: StringName
```

角色也可以是 Resource：

```gdscript
class_name CharacterData
extends Resource

@export var id: StringName
@export var display_name: String
@export var name_color: Color
@export var portraits: Dictionary[StringName, Texture2D]
```

于是职责变成：

- `DialogueLine`：描述一句话
- `CharacterData`：描述角色
- `DialogueRunner`：推进故事状态
- `DialogueBox`：显示文字
- `CharacterDisplay`：显示立绘
- `SceneDirector`：协调这些模块

Resource 很适合可保存、可复用、可在 Inspector 编辑的数据对象。[Resource 文档](https://docs.godotengine.org/en/stable/classes/class_resource.html)

注意：Resource 默认可能被多个实例共享。运行时可变数据不要随意写回资源；必要时使用 `duplicate()`，或者把运行状态保存在单独的 `GameState` 中。

### 6. 克制使用 Autoload

适合 Autoload 的通常只有：

- 当前存档与全局游戏状态
- 设置数据
- 场景切换服务
- 跨场景持续播放的音乐
- 本地化或持久化服务

不适合放进去的包括：

- 某一章的对话控制器
- 当前对话框
- 角色立绘节点
- 某个小游戏的规则
- 所有系统都能随意修改的巨大 `GameManager`

判断标准可以是：

1. 它是否必须跨场景存在？
2. 是否确实只有一个实例？
3. 是否有清晰、较小的公共 API？
4. 是否不会逐渐变成“任何脚本都往里面塞东西”的全局状态仓库？

只有全部接近“是”时才使用 Autoload。

### 7. 少依赖场景树的具体路径

下面这些写法通常是耦合开始增加的信号：

```gdscript
get_parent().get_parent()
get_node("/root/Main/World/UI/DialogueBox")
get_tree().current_scene.get_node("...")
```

更好的选择依次是：

- 同一场景内稳定的子节点：`@onready` / `%UniqueNode`
- 外部依赖：由拥有者注入
- 状态变化通知：信号
- 同类型的一组临时对象：Group
- 真正的全局服务：Autoload

### 8. 给脚本明确类型和较小的公共 API

推荐：

```gdscript
@export var typing_speed: float = 30.0
@onready var text_label: RichTextLabel = %TextLabel

func show_line(line: DialogueLine) -> void:
    ...
```

尽量避免让其他模块直接修改内部成员：

```gdscript
dialogue_box._text_label.text = "..."
```

下划线字段应被看作私有实现。提供 `show_line()`、`clear()`、`set_visible_characters()` 之类的公开方法。这样即使未来替换 UI 节点，外部代码也不需要改变。

### 9. 不要过早设计庞大的架构

小型视觉小说不需要从第一天就引入：

- 通用依赖注入框架
- 全局 EventBus
- ECS
- 多层 Repository/Service/Controller
- 为每种对话命令创建复杂抽象

先分清四类东西通常已经足够：

```text
数据       Resource / JSON / CSV
表现       Control、Sprite2D、AnimationPlayer
规则       普通 GDScript 类或 Node
流程协调   SceneDirector / DialogueRunner
```

只有当同一种变化已经反复造成修改时，再提取抽象。

## 二、推荐学习的开源项目

我最推荐：

### GDQuest — Godot 2D Visual Novel

仓库：[gdquest-demos/godot-4-2d-visual-novel](https://github.com/gdquest-demos/godot-4-2d-visual-novel)

选择它的理由：

- 专门用于演示视觉小说基础，而不是体量庞大的完整引擎
- Godot 4 项目
- 项目规模较小，核心代码容易完整读完
- 包含对白解析、分支选择、人物立绘、背景、音乐、转场和文本框
- 将 `Parser`、`TextBox`、`Characters`、`Backgrounds`、`Autoloads` 等职责分开
- 使用一个 `ScenePlayer` 作为上层流程协调器
- README 明确说明它用于展示视觉小说和对话系统的基础实现；仓库目前约 200 个星标、近百次提交，作为学习样例比随机 game-jam 项目更可靠。[仓库说明](https://github.com/gdquest-demos/godot-4-2d-visual-novel)

许可证需要特别注意：

- 脚本、场景、Shader 等 Godot 生成资源使用 MIT
- 美术资源使用 CC BY-NC-SA 4.0，不能直接拿去做商业项目  
  [项目许可证](https://github.com/gdquest-demos/godot-4-2d-visual-novel/blob/main/LICENSE)

建议按这个顺序阅读：

1. `Main.tscn` 和 `Main.gd`：寻找程序入口。
2. `ScenePlayer.tscn`：先看节点之间的组合关系。
3. `ScenePlayer.gd`：理解故事命令如何分发给文本、角色、背景和音乐模块。
4. `TextBox/`：观察一个相对独立的 UI 子系统。
5. `Parser/`：理解故事文本如何转换为运行时数据。
6. `Autoloads/`：分析哪些状态被作者设计成全局状态。
7. 最后反向检查：哪些依赖通过方法、信号、节点引用或 Autoload 建立。

它是很好的学习项目，但不是“每一行都是最终最佳实践”。例如 `ScenePlayer.gd` 已经承担了较多命令分派职责，而且部分代码访问了其他对象的下划线成员。学习时应同时观察它哪些地方组织得好、哪些地方在继续扩大后应拆成命令处理器或公共接口——这反而是很合适的架构练习。